# 00 · Data Setup & Loading（数据准备与加载）

## 1. 项目目标（Purpose）

本 Notebook 的目标是对电商原始行为数据进行**统一加载、字段理解、清洗与结构化整理**，建立后续所有分析（用户画像、分群、趋势、建模）的**稳定数据底座**。

本阶段不做业务分析，只解决三个问题：

> **数据是什么？是否可信？是否能被后续模块安全复用？**

---

## 2. 数据表概览（Raw Tables Overview）

本项目共使用 **4 张原始数据表**，分别覆盖：
**用户行为、商品属性、商品类目结构** 三个层面。

---

### 2.1 `events.csv` —— 用户行为事件表（核心主表）

**粒度：** 行为级（1 行 = 用户一次行为）

| 字段              | 含义                                   |
| --------------- | ------------------------------------ |
| `timestamp`     | 行为发生时间（Unix 毫秒）                      |
| `visitorid`     | 用户 ID                                |
| `event`         | 行为类型（view / addtocart / transaction） |
| `itemid`        | 商品 ID                                |
| `transactionid` | 交易 ID（仅在 transaction 行为中出现）          |

**关键认知：**

* 同一用户可有多条行为
* `transactionid` 非空 ≈ 下单行为
* **这是后续 LRFM、转化率、漏斗分析的核心数据源**

---

### 2.2 `item_properties_part*.csv` —— 商品属性表（纵表结构）

**粒度：** 商品属性变更级（1 行 = 某商品某时间点的一个属性）

| 字段          | 含义                             |
| ----------- | ------------------------------ |
| `timestamp` | 属性记录时间                         |
| `itemid`    | 商品 ID                          |
| `property`  | 属性类型（如 categoryid、available 等） |
| `value`     | 属性取值（可能为数值 / 字符串 / 多值）         |

**重要特点：**

* **纵表（long format）**，非一行一个商品
* 同一商品同一属性可能随时间变化
* `value` 字段类型高度不统一（需要解析）

---

### 2.3 `category_tree.csv` —— 商品类目层级表

**粒度：** 类目级

| 字段           | 含义               |
| ------------ | ---------------- |
| `categoryid` | 当前类目 ID          |
| `parentid`   | 父类目 ID（为空表示顶级类目） |

**作用：**

* 构建商品的**多级类目路径**
* 支持后续：

  * 按大类 / 子类分析
  * 类目层级聚合（如 GMV by 一级类目）

---

## 3. 数据加载与预处理策略（Loading Strategy）

### 3.1 时间字段统一

* 所有 `timestamp` 均为 **Unix 毫秒**
* 在加载阶段统一转换为 `datetime`
* 明确全数据时间范围，作为后续窗口切分依据

---

### 3.2 主键与粒度确认

| 表名              | 主键                                    | 粒度    |
| --------------- | ------------------------------------- | ----- |
| events          | (visitorid, timestamp, event, itemid) | 行为级   |
| item_properties | (itemid, property, timestamp)         | 属性变更级 |
| category_tree   | categoryid                            | 类目级   |

---

## 4. 数据清洗与质量控制（Data Cleaning & QC）

### 4.1 Events 表清洗重点

* 去除关键字段缺失的记录（visitorid / timestamp）
* 明确行为类型枚举范围（view / addtocart / transaction）
* 对 `transactionid`：

  * 仅在 `event = transaction` 时保留
  * 其他情况设为缺失

---

### 4.2 Item Properties 表处理策略（关键）

由于该表**结构复杂**，本阶段仅做**最小可用处理**：

* 保留与后续分析直接相关的属性：

  * `categoryid`
  * `available`（是否可售）
* 不在本 Notebook 展开宽表
* 为后续 Notebook 提供**干净、可筛选的属性源数据**

> 商品属性的宽表构建与特征工程将在 `01_feature_engineering` 中完成。

---

## 5. 时间边界与数据泄漏控制（Temporal Scope）

* 明确全数据时间跨度（min / max timestamp）
* 本 Notebook 不构造任何标签（label）
* 不使用任何未来信息
* 为后续所有特征工程保留 **时间切片能力**

---

## 6. 清洗后数据输出（Clean Data Outputs）

清洗完成后的数据将统一缓存为 Parquet 格式，作为后续 Notebook 的输入：

```text
data_clean/
├── events_clean.parquet
├── item_properties_clean.parquet
├── category_tree_clean.parquet
```

**设计原则：**

* 所有下游分析只读 `data_clean`
* 保证“单一数据真源（Single Source of Truth）”
* 提高 Notebook 间解耦与复现性

---

## 7. 本 Notebook 的职责边界（Out of Scope）

本阶段**不做**：

* 用户画像 / LRFM 计算
* 用户分群或建模
* 任何业务结论

> 本 Notebook 的唯一职责是：
> **把复杂、异构的原始数据 → 统一、可信、可复用的数据层**




In [12]:
import os 
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r'E:\data_analysis\ecommece')    # assume notebook is in project root
DATA_RAW_DIR = PROJECT_DIR / "data"      # put your csv files here
DATA_CLEAN_DIR = PROJECT_DIR / "data_clean" / "00_setup_and_data_load"  # parquet outputs

DATA_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TIMESTAMP_UNIT = "ms"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_RAW_DIR:", DATA_RAW_DIR)
print("DATA_CLEAN_DIR:", DATA_CLEAN_DIR)

PROJECT_DIR: E:\data_analysis\ecommece
DATA_RAW_DIR: E:\data_analysis\ecommece\data
DATA_CLEAN_DIR: E:\data_analysis\ecommece\data_clean\00_setup_and_data_load


In [13]:
csv_files = list(DATA_RAW_DIR.glob("*.csv"))
csv_files.sort()

print("Step 1) CSV files found:")
for f in csv_files:
    print(" -", f.name)

SAMPLE_ROWS = 50

print("\n")

for f in csv_files:
    print("\n" + "=" * 90)
    print(f"Step 2) Preview file: {f.name}")

    df_sample = pd.read_csv(f, nrows = SAMPLE_ROWS, low_memory = False)

    print("Rows/Cols (sample):", df_sample.shape)
    print("Columns:", list(df_sample.columns))

    display(df_sample.head(3))

    print("Dtypes:")
    print(df_sample.dtypes)

        # Step 3) Special checks: timestamp / event
    if "timestamp" in df_sample.columns:
        ts_dtype = df_sample["timestamp"].dtype
        print("\nCheck) timestamp dtype =", ts_dtype)

        # If timestamp becomes float, it may lose precision. We'll force int64 later.
        if np.issubdtype(ts_dtype, np.floating):
            print("⚠️ Warning: timestamp is float in the sample. Full load will force int64 to avoid precision loss.")

    if "event" in df_sample.columns:
        unique_events = df_sample["event"].dropna().unique()
        print("\nCheck) unique event values (sample):", unique_events)

Step 1) CSV files found:
 - category_tree.csv
 - events.csv
 - item_properties_part1.csv
 - item_properties_part2.csv



Step 2) Preview file: category_tree.csv
Rows/Cols (sample): (50, 2)
Columns: ['categoryid', 'parentid']


,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0


Dtypes:
categoryid      int64
parentid      float64
dtype: object

Step 2) Preview file: events.csv
Rows/Cols (sample): (50, 5)
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN


Dtypes:
timestamp          int64
visitorid          int64
event             object
itemid             int64
transactionid    float64
dtype: object

Check) timestamp dtype = int64

Check) unique event values (sample): ['view' 'addtocart']

Step 2) Preview file: item_properties_part1.csv
Rows/Cols (sample): (50, 4)
Columns: ['timestamp', 'itemid', 'property', 'value']


,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566


Dtypes:
timestamp     int64
itemid        int64
property     object
value        object
dtype: object

Check) timestamp dtype = int64

Step 2) Preview file: item_properties_part2.csv
Rows/Cols (sample): (50, 4)
Columns: ['timestamp', 'itemid', 'property', 'value']


,timestamp,itemid,property,value
0,1433041200000,183478,561,769062
1,1439694000000,132256,976,n26.400 1135780
2,1435460400000,420307,921,1149317 1257525


Dtypes:
timestamp     int64
itemid        int64
property     object
value        object
dtype: object

Check) timestamp dtype = int64


In [17]:
print("Loading category_tree.csv ...")
cat = pd.read_csv(
    DATA_RAW_DIR / "category_tree.csv",
    dtype = {"categoryid": "int64", "parentid": "float64"},
)

cat["parentid"] = cat["parentid"].astype("Int64")

print("category_tree shape:", cat.shape)
print("category_tree null parentid:", cat["parentid"].isna().sum())

cat_out = DATA_CLEAN_DIR / "category_tree_clean.parquet"
cat.to_parquet(cat_out, index=False)
print("Saved:", cat_out)

print("\nLoading events.csv ...")
events = pd.read_csv(
    DATA_RAW_DIR / "events.csv",
    dtype = {
        "timestamp": "int64",
        "visitorid": "int64",
        "event": "string",
        "itemid": "int64",
        "transactionid": "float64",
    },
)

events["dt"] = pd.to_datetime(events["timestamp"], unit = TIMESTAMP_UNIT, utc = True)

events["transactionid"] = events["transactionid"].astype("Int64")

events = events.dropna(subset=["visitorid", "timestamp", "event", "itemid"])  # safety
events = events.drop_duplicates()

events["event"] = events["event"].str.lower().str.strip()

known_events = {"view", "addtocart", "transaction"}
events = events[events["event"].isin(known_events)].copy()

print("events shape:", events.shape)
print("events time range:", events["dt"].min(), "->", events["dt"].max())
print("events unique visitors:", events["visitorid"].nunique())
print("events event counts:\n", events["event"].value_counts())

events_out = DATA_CLEAN_DIR / "events_clean.parquet"
events.to_parquet(events_out, index=False)
print("Saved:", events_out)

print("\nLoading item_properties_part1.csv + part2.csv ...")

prop_dtype = {
    "timestamp": "int64",
    "itemid": "int64",
    "property": "string",
    "value": "string",
}

props_1 = pd.read_csv(DATA_RAW_DIR / "item_properties_part1.csv", dtype=prop_dtype)
props_2 = pd.read_csv(DATA_RAW_DIR / "item_properties_part2.csv", dtype=prop_dtype)

props = pd.concat([props_1, props_2], ignore_index=True)

# convert timestamp -> datetime
props["dt"] = pd.to_datetime(props["timestamp"], unit=TIMESTAMP_UNIT, utc=True)

# basic cleaning
props["property"] = props["property"].str.lower().str.strip()
props["value"] = props["value"].str.strip()

props = props.dropna(subset=["itemid", "timestamp", "property", "value"])
props = props.drop_duplicates()

print("item_properties shape:", props.shape)
print("item_properties time range:", props["dt"].min(), "->", props["dt"].max())
print("unique items:", props["itemid"].nunique())
print("top properties:\n", props["property"].value_counts().head(10))

props_out = DATA_CLEAN_DIR / "item_properties_clean.parquet"
props.to_parquet(props_out, index=False)
print("Saved:", props_out)

print("\nDone. Clean parquet files are ready in:", DATA_CLEAN_DIR)

Loading category_tree.csv ...
category_tree shape: (1669, 2)
category_tree null parentid: 25
Saved: E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\category_tree_clean.parquet

Loading events.csv ...
events shape: (2755641, 6)
events time range: 2015-05-03 03:00:04.384000+00:00 -> 2015-09-18 02:59:47.788000+00:00
events unique visitors: 1407580
events event counts:
 event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: Int64
Saved: E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\events_clean.parquet

Loading item_properties_part1.csv + part2.csv ...
item_properties shape: (20275902, 5)
item_properties time range: 2015-05-10 03:00:00+00:00 -> 2015-09-13 03:00:00+00:00
unique items: 417053
top properties:
 property
888           3000398
790           1790516
available     1503639
categoryid     788214
6              631471
283            597419
776            574220
678            481966
364            476486
202           

In [22]:
cat = pd.read_parquet(DATA_CLEAN_DIR / "category_tree_clean.parquet")
events = pd.read_parquet(DATA_CLEAN_DIR / "events_clean.parquet")
props = pd.read_parquet(DATA_CLEAN_DIR / "item_properties_clean.parquet")

def profile_table(df: pd.DataFrame, name: str, key_cols = None, time_col = None):
    info = {}
    info["table"] = name
    info["rows"] = len(df)
    info["cols"] = df.shape[1]
    info["memory_mb"] = df.memory_usage(deep = True).sum() / (1024 ** 2)

    info["missing_cells"] = int(df.isna().sum().sum())

    if key_cols:
        info["dup_rows_by_key"] = int(df.duplicated(subset=key_cols).sum())
    else:
        info["dup_rows_by_key"] = None

    if time_col and time_col in df.columns:
        info["time_min"] = df[time_col].min()
        info["time_max"] = df[time_col].max()
    else:
        info["time_min"] = None
        info["time_max"] = None

    return info

summary = []
summary.append(profile_table(cat, "category_tree", key_cols=["categoryid"]))
summary.append(profile_table(events, "events", key_cols=["visitorid", "timestamp", "event", "itemid"], time_col="dt"))
summary.append(profile_table(props, "item_properties", key_cols=["itemid", "property", "timestamp", "value"], time_col="dt"))

summary_df = pd.DataFrame(summary)

print("=== Data Summary (core) ===")
display(summary_df)

event_counts = events["event"].value_counts()
total_events = len(events)

funnel_df = pd.DataFrame({
    "event": event_counts.index,
    "count": event_counts.values,
})
funnel_df["share_of_events"] = funnel_df["count"] / total_events

visitor_funnel = events.groupby("event")["visitorid"].nunique().rename("unique_visitors").reset_index()

funnel_df = funnel_df.merge(visitor_funnel, on="event", how="left")
funnel_df["share_of_visitors"] = funnel_df["unique_visitors"] / events["visitorid"].nunique()

print("\n=== Funnel (events + visitors) ===")
display(funnel_df)

key_props = ["categoryid", "available"]
key_props_df = (props[props["property"].isin(key_props)]
                .groupby("property")["itemid"]
                .nunique()
                .rename("unique_items_with_property")
                .reset_index())
key_props_df["share_of_all_items"] = key_props_df["unique_items_with_property"] / props["itemid"].nunique()

print("\n=== Key item properties coverage ===")
display(key_props_df)

=== Data Summary (core) ===


,table,rows,cols,memory_mb,missing_cells,dup_rows_by_key,time_min,time_max
0,category_tree,1669,2,0.027184,25,0,NaT,NaT
1,events,2755641,6,268.533271,2733184,0,2015-05-03 03:00:04.384000+00:00,2015-09-18 02:59:47.788000+00:00
2,item_properties,20275902,5,3080.188096,0,0,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00



=== Funnel (events + visitors) ===


,event,count,share_of_events,unique_visitors,share_of_visitors
0,view,2664218,0.966823,1404179,0.997584
1,addtocart,68966,0.025027,37722,0.026799
2,transaction,22457,0.008149,11719,0.008326



=== Key item properties coverage ===


,property,unique_items_with_property,share_of_all_items
0,available,417053,1.0
1,categoryid,417053,1.0


In [23]:
out_dir = DATA_CLEAN_DIR  # same folder
summary_path = out_dir / "data_summary.parquet"
summary_csv_path = out_dir / "data_summary.csv"
funnel_path = out_dir / "events_funnel.csv"
key_props_path = out_dir / "item_properties_key_coverage.csv"

summary_df.to_parquet(summary_path, index=False)
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
funnel_df.to_csv(funnel_path, index=False, encoding="utf-8-sig")
key_props_df.to_csv(key_props_path, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(" -", summary_path)
print(" -", summary_csv_path)
print(" -", funnel_path)
print(" -", key_props_path)


Saved:
 - E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\data_summary.parquet
 - E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\data_summary.csv
 - E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\events_funnel.csv
 - E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\item_properties_key_coverage.csv
